# Review-to-Revenue Agent

This section connects the reusable restaurant tools to a tool-calling agent. The agent uses an LLM to interpret the user’s question, choose the appropriate tool, call the tool, and synthesize a final answer using the tool results.

---
## 1. Configure LLM Endpoint

Baseline: Llama 3.3 70B
Comparirson: GPT OSS 20b (cheaper model)

In [ ]:
# LLM endpoints used for the restaurant comparison agent

LLAMA_3_3_70B_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"
GPT_OSS_20B_ENDPOINT = "databricks-gpt-oss-20b"

AVAILABLE_LLMS = [
    LLAMA_3_3_70B_ENDPOINT,
    GPT_OSS_20B_ENDPOINT
]

# Choose the default LLM for the first agent run
CHOSEN_LLM_ENDPOINT = LLAMA_3_3_70B_ENDPOINT

print("=" * 60)
print("Configuring LLM endpoints...")
print("=" * 60)

print("Available LLM endpoints:")
for endpoint in AVAILABLE_LLMS:
    print(f"- {endpoint}")

print(f"\nDefault LLM endpoint: {CHOSEN_LLM_ENDPOINT}")

---
## 2. Define Tool Metadata

In [ ]:
import json
from dataclasses import dataclass
from typing import Callable

@dataclass
class ToolInfo:
    name: str
    spec: dict
    exec_fn: Callable

---
## 3. Define Tool Specifications

In [ ]:
# Tool specifications in OpenAI-compatible function-calling format

business_lookup_spec = {
    "type": "function",
    "function": {
        "name": "business_lookup",
        "description": (
            "Retrieve rating, review count, rating distribution, and restaurant metadata "
            "for a specific restaurant."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant to look up."
                },
                "limit": {
                    "type": "integer",
                    "description": "Maximum number of matching restaurants to return."
                }
            },
            "required": ["restaurant_name"]
        }
    }
}


competitor_comparison_spec = {
    "type": "function",
    "function": {
        "name": "competitor_comparison",
        "description": (
            "Compare two specific restaurants by rating, review volume, and rating distribution."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_1": {
                    "type": "string",
                    "description": "Name or partial name of the first restaurant."
                },
                "restaurant_2": {
                    "type": "string",
                    "description": "Name or partial name of the second restaurant."
                },
                "limit_per_name": {
                    "type": "integer",
                    "description": "Maximum number of matches to return per restaurant name."
                }
            },
            "required": ["restaurant_1", "restaurant_2"]
        }
    }
}


theme_retrieval_spec = {
    "type": "function",
    "function": {
        "name": "theme_retrieval",
        "description": (
            "Retrieve reviews related to a user-provided theme using semantic Vector Search "
            "over restaurant review text. Use this for qualitative questions about customer experiences, "
            "including service, food quality, ambiance, price, wait time, complaints, and special occasions."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "restaurant_name": {
                    "type": "string",
                    "description": "Name or partial name of the restaurant."
                },
                "theme_query": {
                    "type": "string",
                    "description": (
                        "Natural-language theme to search for in review text. "
                        "For example: 'service and staff friendliness', "
                        "'pasta quality and authentic Italian food', or "
                        "'ambiance for date night'."
                    )
                },
                "num_results": {
                    "type": "integer",
                    "description": "Number of relevant review excerpts to retrieve."
                }
            },
            "required": ["restaurant_name", "theme_query"]
        }
    }
}

---
## 4. Register Tools

In [ ]:
TOOL_INFOS = [
    ToolInfo(
        name="business_lookup",
        spec=business_lookup_spec,
        exec_fn=business_lookup
    ),
    ToolInfo(
        name="competitor_comparison",
        spec=competitor_comparison_spec,
        exec_fn=competitor_comparison
    ),
    ToolInfo(
        name="theme_retrieval",
        spec=theme_retrieval_spec,
        exec_fn=theme_retrieval
    )
]

print("=" * 60)
print("Registering agent tools...")
print("=" * 60)

print(f"Registered {len(TOOL_INFOS)} tools:")
for tool in TOOL_INFOS:
    print(f"- {tool.name}")

---
## 5. Create System Prompt

In [ ]:
SYSTEM_PROMPT = """
You are a restaurant comparison assistant focused on San Diego Italian restaurants.

Your scope:
- Answer questions about San Diego Italian restaurants using the available restaurant metrics and review-text tools.
- Help users compare restaurants, understand ratings, review counts, rating distributions, and customer review themes.
- Provide recommendations only when they can be grounded in the available tool outputs.

Out-of-scope requests:
- Do not answer questions unrelated to San Diego Italian restaurants.
- Do not provide general homework help, coding help, medical advice, legal advice, financial advice, travel planning, politics, sports, entertainment, or unrelated factual information.
- Do not invent restaurant data, restaurant names, metrics, ratings, review counts, review text, or business recommendations.

If the user asks an out-of-scope question, politely reject it and briefly explain that you can only help with San Diego Italian restaurant comparison questions using the available data.

You have access to three tools:
1. business_lookup: retrieves restaurant-level metrics such as average rating, review count, and rating distribution.
2. competitor_comparison: compares two restaurants using rating, review volume, and rating distribution.
3. theme_retrieval: retrieves semantically relevant review excerpts for qualitative customer-experience themes.

Tool-use rules:
- For structured questions about one restaurant, use business_lookup.
- For structured comparison questions about two restaurants, use competitor_comparison.
- For qualitative questions about customer experiences, use theme_retrieval.
- For broad comparison, recommendation, or “what does one restaurant do better” questions, use both competitor_comparison and theme_retrieval. The structured comparison shows rating and review-count differences, while theme_retrieval provides specific evidence about what customers mention, such as service, food quality, ambiance, authenticity, value, wait time, and special occasions.
- If a tool returns no results, say that the data was not found.

Use the restaurant names provided by the user. Do not assume the user only cares about a fixed pair of restaurants.
Be concise, but include enough evidence from tool outputs to justify the answer.
"""

---
## 6. Create Tool-Calling Agent Class with MLflow Tracing

In [ ]:
import mlflow
from databricks.sdk import WorkspaceClient

# Enable MLflow tracing for OpenAI-compatible chat calls
mlflow.openai.autolog()

print("MLflow OpenAI autologging enabled.")

In [ ]:
class RestaurantToolCallingAgent:
    def __init__(self, llm_endpoint: str, tools: list[ToolInfo]):
        self.llm_endpoint = llm_endpoint
        self.tools = {tool.name: tool for tool in tools}
        self.workspace_client = WorkspaceClient()
        self.client = self.workspace_client.serving_endpoints.get_open_ai_client()
    
    def get_tool_specs(self):
        return [tool.spec for tool in self.tools.values()]
    
    def execute_tool(self, tool_name: str, arguments: dict):
        if tool_name not in self.tools:
            return {
                "status": "error",
                "message": f"Unknown tool: {tool_name}"
            }
        
        tool = self.tools[tool_name]
        return tool.exec_fn(**arguments)
    
    def predict(self, user_question: str, max_tool_rounds: int = 5, verbose: bool = True):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_question}
        ]
        
        for _ in range(max_tool_rounds):
            response = self.client.chat.completions.create(
                model=self.llm_endpoint,
                messages=messages,
                tools=self.get_tool_specs(),
                tool_choice="auto"
            )
            
            assistant_message = response.choices[0].message
            messages.append(assistant_message.model_dump(exclude_none=True))
            
            # If the model does not call a tool, return the final response
            if not assistant_message.tool_calls:
                return assistant_message.content
            
            # Execute tool calls
            for tool_call in assistant_message.tool_calls:
                tool_name = tool_call.function.name
                arguments = json.loads(tool_call.function.arguments or "{}")
                
                if verbose:
                    print(f"Calling tool: {tool_name}")
                    print(f"Arguments: {arguments}")
                
                tool_result = self.execute_tool(tool_name, arguments)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result, default=str)
                })
        
        return "The agent reached the maximum number of tool calls before producing a final answer."

---
## 7. Create Agents for Both LLMs

In [ ]:
AGENT_LLAMA_3_3_70B = RestaurantToolCallingAgent(
    llm_endpoint=LLAMA_3_3_70B_ENDPOINT,
    tools=TOOL_INFOS
)

AGENT_GPT_OSS_20B = RestaurantToolCallingAgent(
    llm_endpoint=GPT_OSS_20B_ENDPOINT,
    tools=TOOL_INFOS
)

# Default agent
AGENT = AGENT_LLAMA_3_3_70B

print("=" * 60)
print("Restaurant comparison agents created.")
print("=" * 60)

print(f"Default agent: {LLAMA_3_3_70B_ENDPOINT}")
print(f"Comparison agent: {GPT_OSS_20B_ENDPOINT}")

---
## 8. Compare Both LLMs on the Same Prompts

#### Prompt 1

In [ ]:
test_question = (
    "How many reviews does Parma Cucina Italiana have, and what is its average rating?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 2

In [ ]:
test_question = (
    "Compare Parma Cucina Italiana and Cesarina. What does Cesarina appear to do better?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 3

In [ ]:
test_question = (
    "What are customers saying about the pasta quality and Italian authenticity at Parma Cucina Italiana?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 4

In [ ]:
test_question = (
    "What is the best recipe for homemade lasagna?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)

#### Prompt 5

In [ ]:
test_question = (
    "Can you summarize reviews for a dentist in New York?"
)

print("=" * 80)
print("Llama 3.3 70B")
print("=" * 80)

llama_answer = AGENT_LLAMA_3_3_70B.predict(test_question)
print(llama_answer)


print("\n" + "=" * 80)
print("GPT OSS 20B")
print("=" * 80)

gpt_oss_answer = AGENT_GPT_OSS_20B.predict(test_question)
print(gpt_oss_answer)